# Apple FY2024–FY2025 with tracemodel

Built in four steps: **dimensions → items scaffold → data quality → values**. Run top to bottom, then change things and rerun. Re-running a definition cell is safe: the same definition is a no-op, and `add_check` replaces a check with the same id.

In [1]:
import tracemodel as tm

model = tm.Model()

## 1. Dimensions

In [2]:
model.add_dimension("period", [
    ("FY2024", "Fiscal 2024", {"end": "2024-09-28"}),
    ("FY2025", "Fiscal 2025", {"end": "2025-09-27"}),
])

<Dimension period (flat): FY2024, FY2025>

Every coord entry is a code, a tuple `(code, description)`, or a tuple `(code, description, attrs)`; in a hierarchy the code is a path.

In [3]:
model.add_dimension("sales_category", tm.Hierarchy([
    ("/", "Total"),
    "/products",
    "/products/iphone",
    "/products/mac",
    "/products/ipad",
    ("/products/wearables", "Wearables, Home and Accessories"),
    "/services",
]))

<Dimension sales_category (hierarchy): /, /products, /products/iphone, /products/mac, /products/ipad, /products/wearables, /services>

In [4]:
model.add_dimension("expense_category", tm.Hierarchy([
    ("/rnd", "Research and development"),
    ("/gna", "Selling, general and administrative"),
]))

<Dimension expense_category (hierarchy): /rnd, /gna>

In [5]:
sales = model.dimensions["sales_category"].hierarchy
sales.children("/products"), sales.description("/products/wearables")

(['/products/iphone',
  '/products/mac',
  '/products/ipad',
  '/products/wearables'],
 'Wearables, Home and Accessories')

## 2. Items scaffold

Each item has dims, a data type (`int` or `float`) and `attrs` for units and scale.

In [6]:
USD_M = {"units": "USD", "scale": "millions"}

model.add_item("net_sales", dims=("sales_category", "period"), dtype=float, attrs=USD_M)
model.add_item("cost_of_sales", dims=("sales_category", "period"), dtype=float, attrs=USD_M)
model.add_item("gross_margin", dims=("sales_category", "period"), dtype=float, attrs=USD_M)

Item(name='gross_margin', dims=('sales_category', 'period'), dtype=<class 'float'>, attrs={'units': 'USD', 'scale': 'millions'}, formula=None)

In [7]:
model.add_item("operating_expense", dims=("expense_category", "period"), dtype=float, attrs=USD_M)

Item(name='operating_expense', dims=('expense_category', 'period'), dtype=<class 'float'>, attrs={'units': 'USD', 'scale': 'millions'}, formula=None)

In [8]:
model.add_item("shares_basic", dims=("period",), dtype=int, attrs={"units": "shares", "scale": "thousands"})
model.add_item("earnings_per_share", dims=("period",), dtype=float, attrs={"units": "USD/share"})

Item(name='earnings_per_share', dims=('period',), dtype=<class 'float'>, attrs={'units': 'USD/share'}, formula=None)

## 3. Data quality

In [9]:
model.add_check("DQ-1",
    coords={"period": tm.ANY},   # run once per period
    expect=abs(
        model["net_sales"]["/"]
        - (model["net_sales"]["/products"] + model["net_sales"]["/services"])
    ) <= 0.2,                    # within 0.2 (USD millions)
)

Check(id='DQ-1', coords={'period': ('FY2024', 'FY2025')}, expect=abs(net_sales['/'] - (net_sales['/products'] + net_sales['/services'])) <= 0.2, description=None, params=(), conditions={}, severity='error', dynamic={'period': tm.ANY})

In [10]:
model

<tracemodel.Model>
Dimensions:
  period
    FY2024   Fiscal 2024   end=2024-09-28
    FY2025   Fiscal 2025   end=2025-09-27
  sales_category (hierarchy)
    /                     Total
    /products
    /products/iphone
    /products/mac
    /products/ipad
    /products/wearables   Wearables, Home and Accessories
    /services
  expense_category (hierarchy)
    /rnd   Research and development
    /gna   Selling, general and administrative
Items:
  net_sales            (sales_category, period)  float  units=USD scale=millions
  cost_of_sales        (sales_category, period)  float  units=USD scale=millions
  gross_margin         (sales_category, period)  float  units=USD scale=millions
  operating_expense    (expense_category, period)  float  units=USD scale=millions
  shares_basic         (period)  int  units=shares scale=thousands
  earnings_per_share   (period)  float  units=USD/share
Checks:
  DQ-1   abs(net_sales['/'] - (net_sales['/products'] + net_sales['/services'])) <= 0.2  per 

## 4. Values: `model.set_value` with coords

Every value needs a source dict: `document`, `url`, `section`; `page` and `quote` are optional.

In [11]:
TENK = "https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm"
SO  = {"document": "Apple 10-K FY2025", "url": TENK,
       "section": "Item 8, Consolidated Statements of Operations"}
CAT = {"document": "Apple 10-K FY2025", "url": TENK,
       "section": "Item 8, Note 2 – Revenue, net sales by category"}
GM  = {"document": "Apple 10-K FY2025", "url": TENK,
       "section": "Item 7, MD&A, Gross Margin table"}

### net_sales: sales_category x period

In [12]:
model.set_value("net_sales", coords={"sales_category": "/", "period": "FY2024"}, value=391_035, source=SO)
model.set_value("net_sales", coords={"sales_category": "/", "period": "FY2025"}, value=416_161, source=SO)
model.set_value("net_sales", coords={"sales_category": "/products", "period": "FY2024"}, value=294_866, source=SO)
model.set_value("net_sales", coords={"sales_category": "/products", "period": "FY2025"}, value=307_003, source=SO)
model.set_value("net_sales", coords={"sales_category": "/services", "period": "FY2024"}, value=96_169, source=SO)
model.set_value("net_sales", coords={"sales_category": "/services", "period": "FY2025"}, value=109_158, source=SO)
model.set_value("net_sales", coords={"sales_category": "/products/iphone", "period": "FY2024"}, value=201_183, source=CAT)
model.set_value("net_sales", coords={"sales_category": "/products/iphone", "period": "FY2025"}, value=209_586, source=CAT)
model.set_value("net_sales", coords={"sales_category": "/products/mac", "period": "FY2024"}, value=29_984, source=CAT)
model.set_value("net_sales", coords={"sales_category": "/products/mac", "period": "FY2025"}, value=33_708, source=CAT)
model.set_value("net_sales", coords={"sales_category": "/products/ipad", "period": "FY2024"}, value=26_694, source=CAT)
model.set_value("net_sales", coords={"sales_category": "/products/ipad", "period": "FY2025"}, value=28_023, source=CAT)
model.set_value("net_sales", coords={"sales_category": "/products/wearables", "period": "FY2024"}, value=37_005, source=CAT)
model.set_value("net_sales", coords={"sales_category": "/products/wearables", "period": "FY2025"}, value=35_686, source=CAT)

### cost_of_sales: sales_category x period (Apple reports /, /products, /services only)

In [13]:
model.set_value("cost_of_sales", coords={"sales_category": "/", "period": "FY2024"}, value=210_352, source=SO)
model.set_value("cost_of_sales", coords={"sales_category": "/", "period": "FY2025"}, value=220_960, source=SO)
model.set_value("cost_of_sales", coords={"sales_category": "/products", "period": "FY2024"}, value=185_233, source=SO)
model.set_value("cost_of_sales", coords={"sales_category": "/products", "period": "FY2025"}, value=194_116, source=SO)
model.set_value("cost_of_sales", coords={"sales_category": "/services", "period": "FY2024"}, value=25_119, source=SO)
model.set_value("cost_of_sales", coords={"sales_category": "/services", "period": "FY2025"}, value=26_844, source=SO)

### gross_margin: sales_category x period

In [14]:
model.set_value("gross_margin", coords={"sales_category": "/", "period": "FY2024"}, value=180_683, source=SO)
model.set_value("gross_margin", coords={"sales_category": "/", "period": "FY2025"}, value=195_201, source=SO)
model.set_value("gross_margin", coords={"sales_category": "/products", "period": "FY2024"}, value=109_633, source=GM)
model.set_value("gross_margin", coords={"sales_category": "/products", "period": "FY2025"}, value=112_887, source=GM)
model.set_value("gross_margin", coords={"sales_category": "/services", "period": "FY2024"}, value=71_050, source=GM)
model.set_value("gross_margin", coords={"sales_category": "/services", "period": "FY2025"}, value=82_314, source=GM)

### operating_expense: expense_category x period

In [15]:
model.set_value("operating_expense", coords={"expense_category": "/rnd", "period": "FY2024"}, value=31_370, source=SO)
model.set_value("operating_expense", coords={"expense_category": "/rnd", "period": "FY2025"}, value=34_550, source=SO)
model.set_value("operating_expense", coords={"expense_category": "/gna", "period": "FY2024"}, value=26_097, source=SO)
model.set_value("operating_expense", coords={"expense_category": "/gna", "period": "FY2025"}, value=27_601, source=SO)

### shares_basic and earnings_per_share: period only

In [16]:
model.set_value("shares_basic", coords={"period": "FY2024"}, value=15_343_783, source=SO)
model.set_value("shares_basic", coords={"period": "FY2025"}, value=14_948_500, source=SO)
model.set_value("earnings_per_share", coords={"period": "FY2024"}, value=6.11, source=SO)
model.set_value("earnings_per_share", coords={"period": "FY2025"}, value=7.49, source=SO)

In [17]:
model

<tracemodel.Model>
Dimensions:
  period
    FY2024   Fiscal 2024   end=2024-09-28
    FY2025   Fiscal 2025   end=2025-09-27
  sales_category (hierarchy)
    /                     Total
    /products
    /products/iphone
    /products/mac
    /products/ipad
    /products/wearables   Wearables, Home and Accessories
    /services
  expense_category (hierarchy)
    /rnd   Research and development
    /gna   Selling, general and administrative
Items:
  net_sales            (sales_category, period)  float  units=USD scale=millions
  cost_of_sales        (sales_category, period)  float  units=USD scale=millions
  gross_margin         (sales_category, period)  float  units=USD scale=millions
  operating_expense    (expense_category, period)  float  units=USD scale=millions
  shares_basic         (period)  int  units=shares scale=thousands
  earnings_per_share   (period)  float  units=USD/share
Checks:
  DQ-1   abs(net_sales['/'] - (net_sales['/products'] + net_sales['/services'])) <= 0.2  per 

## Look at the values

Tables render as real tables in Jupyter; `.show()` does the same from inside a cell.

In [18]:
model.table("net_sales")

sales_category,FY2024,FY2025
/,"391,035","416,161"
/products,"294,866","307,003"
/products/iphone,"201,183","209,586"
/products/mac,"29,984","33,708"
/products/ipad,"26,694","28,023"
/products/wearables,"37,005","35,686"
/services,"96,169","109,158"


In [19]:
model.table("cost_of_sales")

sales_category,FY2024,FY2025
/,"210,352","220,960"
/products,"185,233","194,116"
/products/iphone,,
/products/mac,,
/products/ipad,,
/products/wearables,,
/services,"25,119","26,844"


In [20]:
model.table("operating_expense").show()
model.table("shares_basic").show()

expense_category,FY2024,FY2025
/rnd,"31,370","34,550"
/gna,"26,097","27,601"


,FY2024,FY2025
shares_basic,"15,343,783","14,948,500"


`entry` with full coords gives one entry; with fewer coords, a table of entries.

In [21]:
model.entry("earnings_per_share")

period,value,source,replaced
FY2024,6.11,"Apple 10-K FY2025, Item 8, Consolidated Statements of Operations",0
FY2025,7.49,"Apple 10-K FY2025, Item 8, Consolidated Statements of Operations",0


In [22]:
model.entry("net_sales", period="FY2025")

sales_category,period,value,source,replaced
/,FY2025,"416,161","Apple 10-K FY2025, Item 8, Consolidated Statements of Operations",0
/products,FY2025,"307,003","Apple 10-K FY2025, Item 8, Consolidated Statements of Operations",0
/services,FY2025,"109,158","Apple 10-K FY2025, Item 8, Consolidated Statements of Operations",0
/products/iphone,FY2025,"209,586","Apple 10-K FY2025, Item 8, Note 2 – Revenue, net sales by category",0
/products/mac,FY2025,"33,708","Apple 10-K FY2025, Item 8, Note 2 – Revenue, net sales by category",0
/products/ipad,FY2025,"28,023","Apple 10-K FY2025, Item 8, Note 2 – Revenue, net sales by category",0
/products/wearables,FY2025,"35,686","Apple 10-K FY2025, Item 8, Note 2 – Revenue, net sales by category",0


In [23]:
model.entry("net_sales", sales_category="/", period="FY2025")

Entry(item='net_sales', coords={'sales_category': '/', 'period': 'FY2025'}, value=416161, source={'document': 'Apple 10-K FY2025', 'url': 'https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm', 'section': 'Item 8, Consolidated Statements of Operations'}, history=[], overrides_formula=False)

## Arithmetic on items

Expressions broadcast by dim name and display their values.

In [24]:
model["net_sales"] / model["net_sales"]["/"]   # mix: share of total net sales

sales_category,FY2024,FY2025
/,1,1
/products,0.754065,0.737702
/products/iphone,0.514488,0.503618
/products/mac,0.076679,0.080997
/products/ipad,0.068265,0.067337
/products/wearables,0.094633,0.08575
/services,0.245935,0.262298


In [25]:
model["gross_margin"] / model["net_sales"]     # gross margin %

sales_category,FY2024,FY2025
/,0.462063,0.469052
/products,0.371806,0.367707
/products/iphone,,
/products/mac,,
/products/ipad,,
/products/wearables,,
/services,0.738804,0.754081


In [26]:
model["net_sales"]["/"] * 1000 / model["shares_basic"]   # net sales per share, USD

,FY2024,FY2025
net_sales['/'] * 1000 / shares_basic,25.484915,27.839649


A comparison shows True/False per coord:

In [27]:
model["net_sales"]["/"] == model["net_sales"]["/products"] + model["net_sales"]["/services"]

period,left,right,diff,result
FY2024,"391,035","391,035",0,True
FY2025,"416,161","416,161",0,True


In [28]:
# Declare the item (definition)                                                            
model.add_item("net_income", dims=("period",), dtype=float, attrs=USD_M)                   
                                                                                            
# Enter the values (values)                                                                  
model.set_value("net_income", coords={"period": "FY2024"}, value=93_736, source=SO)          
model.set_value("net_income", coords={"period": "FY2025"}, value=112_010, source=SO)           
                                                                                              
model.table("net_income")              

,FY2024,FY2025
net_income,"93,736","112,010"


In [29]:
model["earnings_per_share"] == model["net_income"] / model["shares_basic"] * 1000

period,left,right,diff,result
FY2024,6.11,6.109054,0.000946,False
FY2025,7.49,7.49306,-0.00306,False


## Run the checks

In [30]:
model.check()

check,coords,result,left,rule,right,diff
DQ-1,period=FY2024,pass,0,<=,0.2,-0.2
DQ-1,period=FY2025,pass,0,<=,0.2,-0.2


In [31]:
print(model.check().explain())

DQ-1  period=FY2024  pass  left 0  <=  right 0.2  diff -0.2
    abs(net_sales['/'] - (net_sales['/products'] + net_sales['/services'])) <= 0.2
      net_sales[sales_category=/, period=FY2024] = 391,035
      net_sales[sales_category=/products, period=FY2024] = 294,866
      net_sales[sales_category=/services, period=FY2024] = 96,169
DQ-1  period=FY2025  pass  left 0  <=  right 0.2  diff -0.2
    abs(net_sales['/'] - (net_sales['/products'] + net_sales['/services'])) <= 0.2
      net_sales[sales_category=/, period=FY2025] = 416,161
      net_sales[sales_category=/products, period=FY2025] = 307,003
      net_sales[sales_category=/services, period=FY2025] = 109,158


Break DQ-1 on purpose: a typo of 1 is outside the 0.2 tolerance.

In [32]:
model.set_value("net_sales", {"sales_category": "/", "period": "FY2024"}, 391_036, source=SO)
model.check()

check,coords,result,left,rule,right,diff
DQ-1,period=FY2024,FAIL,1,<=,0.2,0.8
DQ-1,period=FY2025,pass,0,<=,0.2,-0.2


In [33]:
# put it back; the typo stays in the history
model.set_value("net_sales", {"sales_category": "/", "period": "FY2024"}, 391_035, source=SO)
model.entry("net_sales", sales_category="/", period="FY2024").history

[(391035,
  {'document': 'Apple 10-K FY2025',
   'url': 'https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm',
   'section': 'Item 8, Consolidated Statements of Operations'}),
 (391036,
  {'document': 'Apple 10-K FY2025',
   'url': 'https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm',
   'section': 'Item 8, Consolidated Statements of Operations'})]

Change the check and re-run: `add_check` replaces DQ-1 in place.

In [34]:
# A relative tolerance is just an expression: within 0.05% of the total
model.add_check("DQ-1",
    coords={"period": tm.ANY},
    expect=abs(
        model["net_sales"]["/"]
        - (model["net_sales"]["/products"] + model["net_sales"]["/services"])
    ) <= 0.0005 * model["net_sales"]["/"],
)
model.check()

check,coords,result,left,rule,right,diff
DQ-1,period=FY2024,pass,0,<=,195.5175,-195.5175
DQ-1,period=FY2025,pass,0,<=,208.0805,-208.0805


## Conditions: any comparison

`expect=` takes any comparison (`< <= > >= == !=`), joined with `&` (and), `|` (or), `~` (not). Put parentheses around each comparison.

In [35]:
ns = model["net_sales"]
gm = model["gross_margin"]

# Sign rule: every sales figure is non-negative
model.add_check("SIGN-1",
    coords={"sales_category": tm.ANY, "period": tm.ANY},
    expect=ns >= 0,
)

# Range: total gross margin between 30% and 60% of sales
model.add_check("RANGE-1",
    coords={"period": tm.ANY},
    expect=(gm["/"] / ns["/"] >= 0.30) & (gm["/"] / ns["/"] <= 0.60),
)

model.check()

check,coords,result,left,rule,right,diff
DQ-1,period=FY2024,pass,0,<=,195.5175,-195.5175
DQ-1,period=FY2025,pass,0,<=,208.0805,-208.0805
SIGN-1,"sales_category=/, period=FY2024",pass,"391,035",>=,0,"391,035"
SIGN-1,"sales_category=/, period=FY2025",pass,"416,161",>=,0,"416,161"
SIGN-1,"sales_category=/products, period=FY2024",pass,"294,866",>=,0,"294,866"
SIGN-1,"sales_category=/products, period=FY2025",pass,"307,003",>=,0,"307,003"
SIGN-1,"sales_category=/products/iphone, period=FY2024",pass,"201,183",>=,0,"201,183"
SIGN-1,"sales_category=/products/iphone, period=FY2025",pass,"209,586",>=,0,"209,586"
SIGN-1,"sales_category=/products/mac, period=FY2024",pass,"29,984",>=,0,"29,984"
SIGN-1,"sales_category=/products/mac, period=FY2025",pass,"33,708",>=,0,"33,708"


In [36]:
# Display a condition to see each side and the result per coord
gm["/"] / ns["/"] >= 0.30

period,left,right,diff,result
FY2024,0.462063,0.3,0.162063,True
FY2025,0.469052,0.3,0.169052,True


## Function checks: one rule for every parent

When the rule depends on a label, pass a function. Its parameters are dims in `coords`; it is called once per label and returns that label's condition.

In [37]:
ns = model["net_sales"]
sales = model.dimensions["sales_category"].hierarchy

model.add_check("NS-SUM",
    coords={"sales_category": ["/", "/products"], "period": tm.ANY},
    expect=lambda sales_category: abs(
        ns[sales_category] - sum(ns[c] for c in sales.children(sales_category))
    ) <= 0.2,
    description="Each category = the sum of its children",
)
print(model.check().explain("NS-SUM"))

NS-SUM  sales_category=/, period=FY2024  pass  left 0  <=  right 0.2  diff -0.2
    abs(net_sales['/'] - (net_sales['/products'] + net_sales['/services'])) <= 0.2
      net_sales[sales_category=/, period=FY2024] = 391,035
      net_sales[sales_category=/products, period=FY2024] = 294,866
      net_sales[sales_category=/services, period=FY2024] = 96,169
NS-SUM  sales_category=/, period=FY2025  pass  left 0  <=  right 0.2  diff -0.2
    abs(net_sales['/'] - (net_sales['/products'] + net_sales['/services'])) <= 0.2
      net_sales[sales_category=/, period=FY2025] = 416,161
      net_sales[sales_category=/products, period=FY2025] = 307,003
      net_sales[sales_category=/services, period=FY2025] = 109,158
NS-SUM  sales_category=/products, period=FY2024  pass  left 0  <=  right 0.2  diff -0.2
    abs(net_sales['/products'] - (net_sales['/products/iphone'] + net_sales['/products/mac'] + net_sales['/products/ipad'] + net_sales['/products/wearables'])) <= 0.2
      net_sales[sales_category=/pr

## Earlier and later periods: `lag` and `lead`

A check that lags leaves the first period out of its coords: it has nothing before it.

In [38]:
ns = model["net_sales"]

model.add_check("GROWTH-1",
    coords={"period": model.dimensions["period"].coords[1:]},   # FY2025 only
    expect=ns["/"] > ns["/"].lag("period"),
    description="Total net sales grew",
)
print(model.check().explain("GROWTH-1"))

GROWTH-1  period=FY2025  pass  left 416,161  >  right 391,035  diff 25,126
    net_sales['/'] > net_sales['/'].lag('period')
      net_sales[sales_category=/, period=FY2025] = 416,161
      net_sales[sales_category=/, period=FY2024] = 391,035


In [39]:
# Year-over-year growth; FY2024 is blank, it has no prior year
ns["/"] / ns["/"].lag("period") - 1

,FY2024,FY2025
net_sales['/'] / net_sales['/'].lag('period') - 1,,0.064255


## Rejected entries

In [40]:
def attempt(f):
    try:
        f()
        print("accepted")
    except tm.TraceModelError as e:
        print(type(e).__name__, "-", e)

In [41]:
attempt(lambda: model.set_value("net_sales", coords={"sales_category": "/products/vision_pro", "period": "FY2025"}, value=1, source=SO))

UnknownCoord - '/products/vision_pro' is not a coord of 'sales_category'; declared: /, /products, /products/iphone, /products/mac, /products/ipad, /products/wearables, /services


In [42]:
attempt(lambda: model.set_value("net_sales", coords={"sales_category": "/", "period": "FY2023"}, value=1, source=SO))

UnknownCoord - 'FY2023' is not a coord of 'period'; declared: FY2024, FY2025


In [43]:
attempt(lambda: model.set_value("operating_expense", coords={"sales_category": "/", "period": "FY2025"}, value=1, source=SO))

DimMismatch - operating_expense takes coords for (expense_category, period): missing expense_category; extra sales_category


In [44]:
attempt(lambda: model.set_value("shares_basic", coords={"period": "FY2025"}, value=14_948_500.5, source=SO))

InvalidValue - shares_basic is int; got 14948500.5


In [45]:
attempt(lambda: model.set_value("earnings_per_share", coords={"period": "FY2025"}, value=7.49, source={"document": "10-K"}))

InvalidValue - source: missing url, section


## Save and reload

In [46]:
model.to_jsonl("income_statement.jsonl")
print(open("income_statement.jsonl").readline())

{"line_item": "net_sales", "coords": {"sales_category": "/", "period": "FY2024"}, "value": 391035, "source": {"document": "Apple 10-K FY2025", "url": "https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm", "section": "Item 8, Consolidated Statements of Operations"}}

